# Afterimage Qwen training

Use the Python (afterimage_env) kernel. Configure the run, check the inputs, then execute the individual training cells you want. Training saves a verified checkpoint at each completed epoch.

## Configure the run

In [22]:
from pathlib import Path

ROOT = Path(r"D:\Afterimage_Experiments")
PROJECT = Path(r"C:\Users\moht2\Documents\Afterimage")
BASE_MODEL = ROOT / "assets/qwen3-vl-8b-4bit"
DATASET = ROOT / "datasets/aptos_512"

CONDITION = "r16_on"  # "r8_on", "r16_on", "r32_on", "r8_off", "r16_off", or "r32_off"
DATASET_VERSION = "duplicate_excluded"  # or "previous_split"
SEED_PAIR = 1  # 1, 2, or 3
MODE = "full"  # "pilot" runs a short validation-only test
RUN_TAG = "01"
ACTION = "resume"  # "check", "train", "resume", or "score"
RECOVERY_RUN = None  # Set a saved run folder here to resume or score a different run.

SEEDS = {1: (42, 3407), 2: (43, 3408), 3: (44, 3409)}
KEEP_PC_AWAKE = True


## Shared setup

In [5]:
import json
import sys

sys.dont_write_bytecode = True
sys.path.insert(0, str(ROOT / "code"))
from notebook_runner import run_command

SCRIPT = ROOT / "code/qwen_pilot.py"
RECOVERY_SCRIPT = ROOT / "code/recover_qwen.py"
CONDITIONS = {"r8_on": (8, "on"), "r16_on": (16, "on"), "r32_on": (32, "on"), "r16_off": (16, "off"), "r8_off": (8, "off"), "r32_off": (32, "off")}
if MODE not in {"pilot", "full"} or ACTION not in {"check", "train", "resume", "score"}:
    raise ValueError("Choose a valid MODE and ACTION.")
if DATASET_VERSION not in {"duplicate_excluded", "previous_split"}:
    raise ValueError("Choose one of the two APTOS split versions.")
def run_settings(condition, seed_pair, version=None, mode=None):
    version = DATASET_VERSION if version is None else version
    mode = MODE if mode is None else mode
    if version not in {"duplicate_excluded", "previous_split"}:
        raise ValueError("Choose one of the two APTOS split versions.")
    manifest = DATASET / version / "manifest.json"
    rank, clahe = CONDITIONS[condition]
    trainer_seed, adapter_seed = SEEDS[seed_pair]
    name = f"{mode}_{condition}_{version}_t{trainer_seed}_a{adapter_seed}_{RUN_TAG}"
    folder = ROOT / "runs" / name
    arguments = [
        "--project", PROJECT, "--candidate", manifest, "--base-snapshot", BASE_MODEL,
        "--rank", rank, "--clahe", clahe, "--trainer-seed", trainer_seed, "--adapter-seed", adapter_seed,
    ]
    if mode == "full":
        arguments.append("--full")
    if version == "previous_split":
        arguments += ["--historical-split", "--comparison-manifest", DATASET / "duplicate_excluded/manifest.json"]
    return folder, arguments

def train_run(condition, seed_pair, version=None):
    if ACTION != "train":
        print("Set ACTION = 'train' and rerun Configure and Shared setup to enable this run cell.")
        return
    folder, arguments = run_settings(condition, seed_pair, version)
    print(f"Run folder: {folder}", flush=True)
    log = ROOT / "logs/training" / folder.name / "launch.log"
    if folder.exists():
        saved = folder / "pilot_configuration.json"
        if not saved.is_file():
            raise RuntimeError(f"This folder has no saved configuration: {folder}. Keep it for inspection and use a fresh RUN_TAG to start training.")
        config = json.loads(saved.read_text(encoding="utf-8"))
        rank, clahe = CONDITIONS[condition]
        trainer_seed, adapter_seed = SEEDS[seed_pair]
        manifest = DATASET / (DATASET_VERSION if version is None else version) / "manifest.json"
        expected = (MODE, rank, clahe == "on", trainer_seed, adapter_seed, BASE_MODEL.resolve(), manifest.resolve())
        actual = (config["mode"], config["rank"], config["clahe"], config["training"]["trainer_seed"],
                  config["adapter_seed"], Path(config["base"]).resolve(), Path(config["candidate_manifest"] or "").resolve())
        if actual != expected:
            raise RuntimeError(f"Saved settings differ for {folder}. Use a fresh RUN_TAG for these settings.")
        print("Found the saved run. Checking whether to resume, restart, finish scoring, or skip.", flush=True)
        run_command(RECOVERY_SCRIPT, ["--output", folder], log.with_name("resume.log"), keep_pc_awake=KEEP_PC_AWAKE)
        return
    run_command(SCRIPT, [*arguments, "--output", folder], log, keep_pc_awake=KEEP_PC_AWAKE)

run_dir, args = run_settings(CONDITION, SEED_PAIR)
recovery = Path(RECOVERY_RUN).resolve() if RECOVERY_RUN else run_dir
log_dir = ROOT / "logs/training" / run_dir.name
print(f"Python: {sys.executable}")
print(f"Selected run for checks and reports: {run_dir}")
print(f"Action: {ACTION}")


Python: C:\Users\moht2\.conda\envs\afterimage_env\python.exe
Selected run for checks and reports: D:\Afterimage_Experiments\runs\full_r16_on_duplicate_excluded_t42_a3407_01
Action: train


## Check settings and files

You can repeat this check with the same run tag. Each check saves its files and log separately. It does not start or resume training.

In [6]:
if ACTION == "check":
    from uuid import uuid4

    check_id = f"check-{uuid4().hex}"
    preflight_dir = ROOT / "checks" / run_dir.name / check_id
    check_log = log_dir / "checks" / check_id / "preflight.log"
    print(f"Check folder: {preflight_dir}", flush=True)
    run_command(SCRIPT, [*args, "--output", preflight_dir, "--prepare-only"], check_log)
    print("Input check passed. No training was started.", flush=True)
else:
    print("Set ACTION = 'check' to write the input check.")

Set ACTION = 'check' to write the input check.


## Training cells

Set ACTION to train, rerun Configure and Shared setup, then execute only the run cells you want. If you rerun a cell with the same tag, it checks the saved run: it continues from a verified checkpoint, restarts an interrupted run without one, finishes scoring if needed, or skips verified results.

Start with the first 12 cells on the duplicate-excluded split. Then run the six additional CLAHE-off cells to complete the three ranks by two CLAHE settings. The three previous-split cells compare training and validation on the recovered original split against the rank-16 CLAHE-on runs above. Both versions score the same 361 test photos. The previous split retains the original overlaps.

The window from 8 AM on 12 November through 5 PM on 13 November is 33 hours. That fits 11 runs at three hours each, 13 at two and a half hours, or 16 at two hours, before extra turnaround time. The run cells execute when you select them. Copy a run cell and choose its condition and seed pair to extend the notebook. Use a fresh RUN_TAG to rerun the same settings.

### First 12 comparisons on the duplicate-excluded split

In [7]:
# Rank 16, CLAHE on, seed pair 1
train_run("r16_on", 1, "duplicate_excluded")

Run folder: D:\Afterimage_Experiments\runs\full_r16_on_duplicate_excluded_t42_a3407_01
Found the saved run. Checking whether to resume, restart, finish scoring, or skip.
Log: D:\Afterimage_Experiments\logs\training\full_r16_on_duplicate_excluded_t42_a3407_01\resume.log
Preserved partial files in D:\Afterimage_Experiments\runs\full_r16_on_duplicate_excluded_t42_a3407_01\recovery_artifacts\partial-kw0pnalg
No resumable epoch checkpoint. Starting again from the base in a new attempt.
Starting train. Log: D:\Afterimage_Experiments\runs\full_r16_on_duplicate_excluded_t42_a3407_01\attempts\20261112T062021Z-siixd90r\train.log
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
W1112 09:20:27.007000 13700 site-packages\torch\utils\_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1112 09:

In [8]:
# Rank 8, CLAHE on, seed pair 1
train_run("r8_on", 1, "duplicate_excluded")

Run folder: D:\Afterimage_Experiments\runs\full_r8_on_duplicate_excluded_t42_a3407_01
Log: D:\Afterimage_Experiments\logs\training\full_r8_on_duplicate_excluded_t42_a3407_01\launch.log
Prepared {'train': 2786, 'val': 357, 'test': 361} APTOS images in D:\Afterimage_Experiments\runs\full_r8_on_duplicate_excluded_t42_a3407_01
Starting train. Log: D:\Afterimage_Experiments\runs\full_r8_on_duplicate_excluded_t42_a3407_01\attempts\20261112T071021Z-ikizce_t\train.log
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
W1112 10:10:27.015000 23172 site-packages\torch\utils\_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1112 10:10:27.065000 23172 site-packages\torch\utils\_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as a

In [9]:
# Rank 32, CLAHE on, seed pair 1
train_run("r32_on", 1, "duplicate_excluded")

Run folder: D:\Afterimage_Experiments\runs\full_r32_on_duplicate_excluded_t42_a3407_01
Log: D:\Afterimage_Experiments\logs\training\full_r32_on_duplicate_excluded_t42_a3407_01\launch.log
Prepared {'train': 2786, 'val': 357, 'test': 361} APTOS images in D:\Afterimage_Experiments\runs\full_r32_on_duplicate_excluded_t42_a3407_01
Starting train. Log: D:\Afterimage_Experiments\runs\full_r32_on_duplicate_excluded_t42_a3407_01\attempts\20261112T080033Z-xfgbgrof\train.log
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
W1112 11:00:38.179000 29568 site-packages\torch\utils\_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1112 11:00:38.228000 29568 site-packages\torch\utils\_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile 

In [10]:
# Rank 16, CLAHE off, seed pair 1
train_run("r16_off", 1, "duplicate_excluded")

Run folder: D:\Afterimage_Experiments\runs\full_r16_off_duplicate_excluded_t42_a3407_01
Log: D:\Afterimage_Experiments\logs\training\full_r16_off_duplicate_excluded_t42_a3407_01\launch.log
Prepared {'train': 2786, 'val': 357, 'test': 361} APTOS images in D:\Afterimage_Experiments\runs\full_r16_off_duplicate_excluded_t42_a3407_01
Starting train. Log: D:\Afterimage_Experiments\runs\full_r16_off_duplicate_excluded_t42_a3407_01\attempts\20261112T085317Z-93sv6kg1\train.log
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
W1112 11:53:23.030000 10272 site-packages\torch\utils\_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1112 11:53:23.087000 10272 site-packages\torch\utils\_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.comp

In [11]:
# Rank 16, CLAHE on, seed pair 2
train_run("r16_on", 2, "duplicate_excluded")

Run folder: D:\Afterimage_Experiments\runs\full_r16_on_duplicate_excluded_t43_a3408_01
Log: D:\Afterimage_Experiments\logs\training\full_r16_on_duplicate_excluded_t43_a3408_01\launch.log
Prepared {'train': 2786, 'val': 357, 'test': 361} APTOS images in D:\Afterimage_Experiments\runs\full_r16_on_duplicate_excluded_t43_a3408_01
Starting train. Log: D:\Afterimage_Experiments\runs\full_r16_on_duplicate_excluded_t43_a3408_01\attempts\20261112T103151Z-q8pkehpq\train.log
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
W1112 13:31:57.595000 30860 site-packages\torch\utils\_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1112 13:31:57.653000 30860 site-packages\torch\utils\_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile 

In [12]:
# Rank 8, CLAHE on, seed pair 2
train_run("r8_on", 2, "duplicate_excluded")

Run folder: D:\Afterimage_Experiments\runs\full_r8_on_duplicate_excluded_t43_a3408_01
Log: D:\Afterimage_Experiments\logs\training\full_r8_on_duplicate_excluded_t43_a3408_01\launch.log
Prepared {'train': 2786, 'val': 357, 'test': 361} APTOS images in D:\Afterimage_Experiments\runs\full_r8_on_duplicate_excluded_t43_a3408_01
Starting train. Log: D:\Afterimage_Experiments\runs\full_r8_on_duplicate_excluded_t43_a3408_01\attempts\20261112T125246Z-_eyj5tfm\train.log
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
W1112 15:52:52.170000 14220 site-packages\torch\utils\_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1112 15:52:52.219000 14220 site-packages\torch\utils\_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as a

In [13]:
# Rank 32, CLAHE on, seed pair 2
train_run("r32_on", 2, "duplicate_excluded")

Run folder: D:\Afterimage_Experiments\runs\full_r32_on_duplicate_excluded_t43_a3408_01
Log: D:\Afterimage_Experiments\logs\training\full_r32_on_duplicate_excluded_t43_a3408_01\launch.log
Prepared {'train': 2786, 'val': 357, 'test': 361} APTOS images in D:\Afterimage_Experiments\runs\full_r32_on_duplicate_excluded_t43_a3408_01
Starting train. Log: D:\Afterimage_Experiments\runs\full_r32_on_duplicate_excluded_t43_a3408_01\attempts\20261112T141121Z-a51659fa\train.log
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
W1112 17:11:27.162000 31156 site-packages\torch\utils\_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1112 17:11:27.213000 31156 site-packages\torch\utils\_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile 

In [14]:
# Rank 16, CLAHE off, seed pair 2
train_run("r16_off", 2, "duplicate_excluded")

Run folder: D:\Afterimage_Experiments\runs\full_r16_off_duplicate_excluded_t43_a3408_01
Log: D:\Afterimage_Experiments\logs\training\full_r16_off_duplicate_excluded_t43_a3408_01\launch.log
Prepared {'train': 2786, 'val': 357, 'test': 361} APTOS images in D:\Afterimage_Experiments\runs\full_r16_off_duplicate_excluded_t43_a3408_01
Starting train. Log: D:\Afterimage_Experiments\runs\full_r16_off_duplicate_excluded_t43_a3408_01\attempts\20261112T150159Z-hln1gm08\train.log
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
W1112 18:02:05.150000 12100 site-packages\torch\utils\_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1112 18:02:05.201000 12100 site-packages\torch\utils\_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.comp

In [15]:
# Rank 16, CLAHE on, seed pair 3
train_run("r16_on", 3, "duplicate_excluded")

Run folder: D:\Afterimage_Experiments\runs\full_r16_on_duplicate_excluded_t44_a3409_01
Log: D:\Afterimage_Experiments\logs\training\full_r16_on_duplicate_excluded_t44_a3409_01\launch.log
Prepared {'train': 2786, 'val': 357, 'test': 361} APTOS images in D:\Afterimage_Experiments\runs\full_r16_on_duplicate_excluded_t44_a3409_01
Starting train. Log: D:\Afterimage_Experiments\runs\full_r16_on_duplicate_excluded_t44_a3409_01\attempts\20261112T160611Z-_jn8g8fo\train.log
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
W1112 19:06:16.703000 10244 site-packages\torch\utils\_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1112 19:06:16.762000 10244 site-packages\torch\utils\_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile 

In [16]:
# Rank 8, CLAHE on, seed pair 3
train_run("r8_on", 3, "duplicate_excluded")

Run folder: D:\Afterimage_Experiments\runs\full_r8_on_duplicate_excluded_t44_a3409_01
Log: D:\Afterimage_Experiments\logs\training\full_r8_on_duplicate_excluded_t44_a3409_01\launch.log
Prepared {'train': 2786, 'val': 357, 'test': 361} APTOS images in D:\Afterimage_Experiments\runs\full_r8_on_duplicate_excluded_t44_a3409_01
Starting train. Log: D:\Afterimage_Experiments\runs\full_r8_on_duplicate_excluded_t44_a3409_01\attempts\20261112T165617Z-o4l5t6jv\train.log
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
W1112 19:56:23.088000 25328 site-packages\torch\utils\_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1112 19:56:23.145000 25328 site-packages\torch\utils\_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as a

In [17]:
# Rank 32, CLAHE on, seed pair 3
train_run("r32_on", 3, "duplicate_excluded")

Run folder: D:\Afterimage_Experiments\runs\full_r32_on_duplicate_excluded_t44_a3409_01
Log: D:\Afterimage_Experiments\logs\training\full_r32_on_duplicate_excluded_t44_a3409_01\launch.log
Prepared {'train': 2786, 'val': 357, 'test': 361} APTOS images in D:\Afterimage_Experiments\runs\full_r32_on_duplicate_excluded_t44_a3409_01
Starting train. Log: D:\Afterimage_Experiments\runs\full_r32_on_duplicate_excluded_t44_a3409_01\attempts\20261112T180026Z-m06odshh\train.log
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
W1112 21:00:31.421000 3440 site-packages\torch\utils\_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1112 21:00:31.471000 3440 site-packages\torch\utils\_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as

In [18]:
# Rank 16, CLAHE off, seed pair 3
train_run("r16_off", 3, "duplicate_excluded")

Run folder: D:\Afterimage_Experiments\runs\full_r16_off_duplicate_excluded_t44_a3409_01
Log: D:\Afterimage_Experiments\logs\training\full_r16_off_duplicate_excluded_t44_a3409_01\launch.log
Prepared {'train': 2786, 'val': 357, 'test': 361} APTOS images in D:\Afterimage_Experiments\runs\full_r16_off_duplicate_excluded_t44_a3409_01
Starting train. Log: D:\Afterimage_Experiments\runs\full_r16_off_duplicate_excluded_t44_a3409_01\attempts\20261112T190533Z-g_3sdkyw\train.log
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
W1112 22:05:38.646000 23324 site-packages\torch\utils\_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1112 22:05:38.698000 23324 site-packages\torch\utils\_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.comp

### Complete the CLAHE comparison at ranks 8 and 32

In [19]:
# Rank 8, CLAHE off, seed pair 1
train_run("r8_off", 1, "duplicate_excluded")

Run folder: D:\Afterimage_Experiments\runs\full_r8_off_duplicate_excluded_t42_a3407_01
Log: D:\Afterimage_Experiments\logs\training\full_r8_off_duplicate_excluded_t42_a3407_01\launch.log
Prepared {'train': 2786, 'val': 357, 'test': 361} APTOS images in D:\Afterimage_Experiments\runs\full_r8_off_duplicate_excluded_t42_a3407_01
Starting train. Log: D:\Afterimage_Experiments\runs\full_r8_off_duplicate_excluded_t42_a3407_01\attempts\20261112T202454Z-qrsd4xvb\train.log
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
W1112 23:24:59.327000 21616 site-packages\torch\utils\_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1112 23:24:59.376000 21616 site-packages\torch\utils\_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile 

KeyboardInterrupt: 

In [ ]:
# Rank 32, CLAHE off, seed pair 1
train_run("r32_off", 1, "duplicate_excluded")

In [ ]:
# Rank 8, CLAHE off, seed pair 2
train_run("r8_off", 2, "duplicate_excluded")

In [ ]:
# Rank 32, CLAHE off, seed pair 2
train_run("r32_off", 2, "duplicate_excluded")

In [ ]:
# Rank 8, CLAHE off, seed pair 3
train_run("r8_off", 3, "duplicate_excluded")

In [ ]:
# Rank 32, CLAHE off, seed pair 3
train_run("r32_off", 3, "duplicate_excluded")

### Previous-split comparisons

These are three additional rank-16 CLAHE-on runs, matched to the same seed pairs and test IDs.

In [ ]:
# Rank 16, CLAHE on, previous split, seed pair 1
train_run("r16_on", 1, "previous_split")

In [ ]:
# Rank 16, CLAHE on, previous split, seed pair 2
train_run("r16_on", 2, "previous_split")

In [ ]:
# Rank 16, CLAHE on, previous split, seed pair 3
train_run("r16_on", 3, "previous_split")

## Resume or restart a stopped run

Select the saved run in RECOVERY_RUN and set ACTION to resume. The handler resumes a verified checkpoint, restarts from the base if training stopped before a checkpoint, or scores a saved adapter. Each attempt keeps its own logs.

In [23]:
if ACTION == "resume":
    run_command(RECOVERY_SCRIPT, ["--output", recovery], ROOT / "logs/training" / recovery.name / "resume.log", keep_pc_awake=KEEP_PC_AWAKE)
else:
    print("Set ACTION = 'resume' and select the saved run folder.")

Log: D:\Afterimage_Experiments\logs\training\full_r16_on_duplicate_excluded_t42_a3407_01\resume.log
Training and scoring already verified: D:\Afterimage_Experiments\runs\full_r16_on_duplicate_excluded_t42_a3407_01


## Retry scoring the saved adapter

In [ ]:
if ACTION == "score":
    run_command(SCRIPT, ["--output", recovery, "--score-only"], ROOT / "logs/training" / recovery.name / "score_only.log", keep_pc_awake=KEEP_PC_AWAKE)
else:
    print("Set ACTION = 'score' and select the saved run folder.")

## Read results and logs

Each run keeps its split files, checkpoints, selected adapter and predictions. Each attempt keeps a separate system spec file, utilization CSV and stage logs.

In [24]:
folder = recovery if ACTION in {"resume", "score"} else run_dir
report = folder / "run_report.md"
if report.is_file():
    print(report.read_text(encoding="utf-8"))
latest = folder / "latest_attempt.json"
if latest.is_file():
    attempt = Path(json.loads(latest.read_text(encoding="utf-8"))["path"])
    for name in ("train.log", "score.log"):
        path = attempt / name
        if path.is_file():
            print(path)
            print("\n".join(path.read_text(encoding="utf-8", errors="replace").splitlines()[-12:]))
else:
    print(f"No attempt recorded in {folder}.")

# Qwen run report

Status: **passed**.

Fresh APTOS comparison. APTOS photos: 2786 train, 357 val, 361 test.
Rank 16, CLAHE on, trainer seed 42, adapter seed 3407.
Attempt files: D:\Afterimage_Experiments\runs\full_r16_on_duplicate_excluded_t42_a3407_01\attempts\20261112T062021Z-siixd90r

- train: exit 0, 2693.24 s
- train: 534 samples, mean GPU 87.1%, mean CPU 14.6%, peak device VRAM 11448 MiB, peak system RAM 24.20 GiB
- score: exit 0, 298.76 s
- score: 60 samples, mean GPU 66.8%, mean CPU 52.6%, peak device VRAM 12126 MiB, peak system RAM 23.24 GiB

Device and host samples are in utilization.csv. Framework peaks are in the stage result JSONs.

APTOS test: 54.57% accuracy, 0.2807 QWK, 0.8493 macro one-vs-rest ROC AUC.

D:\Afterimage_Experiments\runs\full_r16_on_duplicate_excluded_t42_a3407_01\attempts\20261112T062021Z-siixd90r\train.log
  warnings.warn(
Unsloth: Restored added_tokens_decoder metadata in D:\Afterimage_Experiments\runs\full_r16_on_duplicate_excluded_t42_a3407_01\checkp

## Check completion for the current run tag

Run this cell to check the 21 planned full Qwen runs. It reads the saved run files and checks the adapter and prediction fingerprints. See RUN_CHECKLIST.md for the remaining model and external comparison work.

In [20]:
import runpy

review = runpy.run_path(str(SCRIPT))
worker = review["load_worker"](PROJECT / "experiments/qwen_experiment_worker.py")
def status(folder, condition, seed_pair, version):
    if not folder.exists():
        return "pending"
    config = folder / "pilot_configuration.json"
    score_path = folder / "score_result.json"
    latest = folder / "latest_attempt.json"
    try:
        if not config.is_file():
            return "check setup"
        cfg = json.loads(config.read_text(encoding="utf-8"))
        rank, clahe = CONDITIONS[condition]
        seeds = (cfg["training"]["trainer_seed"], cfg["adapter_seed"])
        expected_manifest = DATASET / version / "manifest.json"
        if (review["identity"](cfg) != cfg.get("identity") or cfg.get("mode") != "full"
                or cfg["rank"] != rank or cfg["clahe"] != (clahe == "on")
                or seeds != SEEDS[seed_pair]
                or Path(cfg["candidate_manifest"]).resolve() != expected_manifest.resolve()
                or worker.sha(expected_manifest) != cfg["candidate_manifest_sha256"]
                or cfg.get("historical_split") != (version == "previous_split")):
            return "check saved settings"
        if not score_path.is_file():
            if (folder / "train_result.json").is_file():
                return "training saved, scoring pending"
            if list((folder / "checkpoints").glob("checkpoint-*/checkpoint_identity.json")):
                return "checkpoint saved"
            return "training incomplete"
        score = json.loads(score_path.read_text(encoding="utf-8"))
        trained = review["trained_adapter"](folder, cfg, worker)
        common_manifest = DATASET / "duplicate_excluded/manifest.json"
        common = json.loads(common_manifest.read_text(encoding="utf-8"))["branches"][clahe]["test"]
        saved_test = folder / "aptos_test.csv"
        common_test = Path(common["path"])
        if worker.sha(saved_test) != cfg["split_sha256"]["test"] or worker.sha(common_test) != common["sha256"]:
            return "check test lists"
        def test_ids(path):
            return [(row["id_code"], row["label"]) for row in worker.read_csv(path)]
        if test_ids(saved_test) != test_ids(common_test):
            return "check common test IDs"
        predictions = folder / "aptos_test.predictions.csv"
        scored_ids = [(row["image_id"], row["y_true"]) for row in worker.read_csv(predictions)]
        if scored_ids != test_ids(common_test):
            return "check prediction IDs"
        result = json.loads(latest.read_text(encoding="utf-8"))
        attempt = Path(result["path"])
        if (score.get("configuration_identity") != cfg["identity"] or score["adapter"] != trained["adapter"]
                or worker.sha(predictions) != score["predictions_sha256"]
                or score["metrics"]["samples"] != 361 or result["status"] != "passed"
                or not (attempt / "utilization.csv").is_file() or not (attempt / "run_report.md").is_file()):
            return "check scores or logs"
        return "verified complete"
    except (OSError, KeyError, TypeError, ValueError, worker.ConditionFailure) as error:
        return f"check artifacts ({type(error).__name__})"

planned = [(condition, seed, "duplicate_excluded") for seed in (1, 2, 3)
           for condition in ("r16_on", "r8_on", "r32_on", "r16_off")]
planned += [(condition, seed, "duplicate_excluded") for seed in (1, 2, 3)
            for condition in ("r8_off", "r32_off")]
planned += [("r16_on", seed, "previous_split") for seed in (1, 2, 3)]
complete = 0
for index, (condition, seed, version) in enumerate(planned, 1):
    folder, _ = run_settings(condition, seed, version, mode="full")
    state = status(folder, condition, seed, version)
    done = state == "verified complete"
    complete += done
    print(f"[{'x' if done else ' '}] {index:02d} {condition} | seed {seed} | {version} | {state}")
print(f"{complete}/{len(planned)} Qwen runs verified for tag {RUN_TAG}.")

[x] 01 r16_on | seed 1 | duplicate_excluded | verified complete
[x] 02 r8_on | seed 1 | duplicate_excluded | verified complete
[x] 03 r32_on | seed 1 | duplicate_excluded | verified complete
[x] 04 r16_off | seed 1 | duplicate_excluded | verified complete
[x] 05 r16_on | seed 2 | duplicate_excluded | verified complete
[x] 06 r8_on | seed 2 | duplicate_excluded | verified complete
[x] 07 r32_on | seed 2 | duplicate_excluded | verified complete
[x] 08 r16_off | seed 2 | duplicate_excluded | verified complete
[x] 09 r16_on | seed 3 | duplicate_excluded | verified complete
[x] 10 r8_on | seed 3 | duplicate_excluded | verified complete
[x] 11 r32_on | seed 3 | duplicate_excluded | verified complete
[x] 12 r16_off | seed 3 | duplicate_excluded | verified complete
[ ] 13 r8_off | seed 1 | duplicate_excluded | training incomplete
[ ] 14 r32_off | seed 1 | duplicate_excluded | pending
[ ] 15 r8_off | seed 2 | duplicate_excluded | pending
[ ] 16 r32_off | seed 2 | duplicate_excluded | pending
[ 